# Limited hyperparameter tuning of the image residual branch (DINOv2, validation only)


In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [7]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal_residual import load_frozen_baseline, FrozenImageOnlyResidualPredictor

SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'dinov2'
IMG_FEAT_DIM = 768

# Search candidates.
PROJ_DIMS = [4, 8, 16]
DROPOUTS = [0.1, 0.3, 0.5]
LRS = [5e-5, 1e-4, 3e-4]
WDS = [0.0, 1e-5, 1e-4]

DEFAULT_PROJ_DIM = 4
DEFAULT_DROPOUT = 0.3
DEFAULT_LR = 1e-4
DEFAULT_WD = 1e-5

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '6_best_image_encoder'
EXP2_CHECKPOINT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'checkpoints' / 'gated_cgm_time'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'tune_dinov2'
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoints'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BASE_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal' / 'results' / 'gated_cgm_time'
BASE_STUB = 'gated_cgm_time'

print(f'ENCODER_NAME={ENCODER_NAME}  IMG_FEAT_DIM={IMG_FEAT_DIM}')


ENCODER_NAME=dinov2  IMG_FEAT_DIM=768


In [8]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
sample_batch = next(iter(train_loader))
print('batch shapes:', [tuple(x.shape) for x in sample_batch])
print('This notebook uses train + validation only; the held-out test set is not loaded.')


def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    pred = model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device))
    return pred, target.to(device)


batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]
This notebook uses train + validation only; the held-out test set is not loaded.


In [4]:
def config_id(img_proj_dim, dropout, lr, weight_decay):
    return f'dinov2_proj{img_proj_dim}_dropout{dropout}_lr{lr}_wd{weight_decay}'


def run_or_load_config(img_proj_dim, dropout, lr, weight_decay, seeds):
    cid = config_id(img_proj_dim, dropout, lr, weight_decay)
    seed_results = {}
    n_trained = 0

    for seed in seeds:
        result_path = RESULTS_DIR / f'{cid}_seed{seed}_results.json'

        if result_path.exists():
            with open(result_path) as f:
                seed_results[seed] = json.load(f)
            continue

        base_model = load_frozen_baseline(seed, EXP2_CHECKPOINT_DIR)
        train.set_seed(seed)  # before constructing the trainable residual branch
        model = FrozenImageOnlyResidualPredictor(
            base_model=base_model, img_feat_dim=IMG_FEAT_DIM, img_proj_dim=img_proj_dim,
            img_dropout=dropout,
        )

        model, history = train.train_model(
            model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
            lr=lr, weight_decay=weight_decay, forward_fn=forward_fn,
            checkpoint_dir=CHECKPOINT_DIR / f'{cid}_seed{seed}',
            verbose=False, progress_bar=True, progress_desc=f'{cid} seed={seed}',
        )

        y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
        val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
        val_results = {str(h): v for h, v in val_results.items()}

        result = {
            'model': f'DINOv2 image-only residual ({cid})',
            'seed': seed, 'encoder': ENCODER_NAME,
            'img_proj_dim': img_proj_dim, 'dropout': dropout, 'lr': lr, 'weight_decay': weight_decay,
            'best_epoch': len(history['train_loss']),
            'val_results': val_results,
        }
        with open(result_path, 'w') as f:
            json.dump(result, f, indent=2)
        seed_results[seed] = result
        n_trained += 1

    return seed_results, n_trained


def evaluate_config(baseline_by_h, seed_results, horizons, seeds):
    per_h = {}
    for h in horizons:
        base = baseline_by_h[h]
        cand = np.asarray([float(seed_results[seed]['val_results'][str(h)]['rmse']) for seed in seeds])
        improvement_pct = (base - cand) / base * 100.0
        wins = int(np.sum(cand < base))
        per_h[h] = {'improvement_pct_mean': float(np.mean(improvement_pct)), 'wins': wins,
                    'n_seeds': len(seeds)}
    mean_improvement = float(np.mean([per_h[h]['improvement_pct_mean'] for h in horizons]))
    total_wins = sum(per_h[h]['wins'] for h in horizons)
    total_pairs = sum(per_h[h]['n_seeds'] for h in horizons)
    return {'per_horizon': per_h, 'mean_improvement_pct': mean_improvement,
            'total_wins': total_wins, 'total_pairs': total_pairs,
            'win_rate': total_wins / total_pairs}


def load_seed_results(result_dir, stub, seeds, result_key='val_results'):
    loaded = {}
    for seed in seeds:
        path = result_dir / f'{stub}_seed{seed}_results.json'
        if not path.exists():
            raise FileNotFoundError(f'Missing result file for seed {seed}: {path}')
        with open(path) as f:
            result = json.load(f)
        if result_key not in result:
            raise KeyError(f'{path.name} missing "{result_key}". Available: {list(result.keys())}')
        loaded[seed] = result
    return loaded


base_loaded = load_seed_results(BASE_DIR, BASE_STUB, SEEDS)
base_by_h = {h: np.asarray([float(base_loaded[seed]['val_results'][str(h)]['rmse']) for seed in SEEDS])
             for h in HORIZONS}
print(f'Loaded CGM+time baseline ({len(SEEDS)} seeds).')

Loaded CGM+time baseline (10 seeds).


In [5]:
EXISTING_DIM4_DIR = PROJECT_ROOT / 'experiments' / '5_residual_multimodal' / 'results' / 'residual_dinov2'
EXISTING_DIM4_STUB = 'residual_dinov2_dim4'

anchor_id = config_id(DEFAULT_PROJ_DIM, DEFAULT_DROPOUT, DEFAULT_LR, DEFAULT_WD)
n_bootstrapped = 0
for seed in SEEDS:
    dest_path = RESULTS_DIR / f'{anchor_id}_seed{seed}_results.json'
    if dest_path.exists():
        continue
    src_path = EXISTING_DIM4_DIR / f'{EXISTING_DIM4_STUB}_seed{seed}_results.json'
    with open(src_path) as f:
        result = json.load(f)
    with open(dest_path, 'w') as f:
        json.dump(result, f, indent=2)
    n_bootstrapped += 1

print(f'Bootstrapped {n_bootstrapped}/{len(SEEDS)} seeds for the default configuration '
      f'({anchor_id}) from the encoder-selection run.')


Bootstrapped 10/10 seeds for the default configuration (dinov2_proj4_dropout0.3_lr0.0001_wd1e-05) from the encoder-selection run.


In [9]:
print('=== Stage 1: img_proj_dim sweep (dropout=0.3, lr=1e-4, weight_decay=1e-5 fixed) ===')

stage1_evals = {}
for proj_dim in PROJ_DIMS:
    seed_results, n_trained = run_or_load_config(proj_dim, DEFAULT_DROPOUT, DEFAULT_LR, DEFAULT_WD, SEEDS)
    ev = evaluate_config(base_by_h, seed_results, HORIZONS, SEEDS)
    stage1_evals[proj_dim] = ev
    print(f"img_proj_dim={proj_dim:>3}  (trained {n_trained}/{len(SEEDS)} new seeds)  "
          f"mean improvement={ev['mean_improvement_pct']:+.3f}%  "
          f"win-rate={ev['total_wins']}/{ev['total_pairs']} ({ev['win_rate']:.1%})")

BEST_PROJ_DIM = max(stage1_evals, key=lambda k: stage1_evals[k]['mean_improvement_pct'])
print(f'\nStage 1 winner: img_proj_dim = {BEST_PROJ_DIM}')


=== Stage 1: img_proj_dim sweep (dropout=0.3, lr=1e-4, weight_decay=1e-5 fixed) ===
img_proj_dim=  4  (trained 0/10 new seeds)  mean improvement=+0.630%  win-rate=50/50 (100.0%)
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.3_lr0.0001_wd1e-05 seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.3_lr0.0001_wd1e-05 seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


...

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,038,922


dinov2_proj16_dropout0.3_lr0.0001_wd1e-05 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,038,922


dinov2_proj16_dropout0.3_lr0.0001_wd1e-05 seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

img_proj_dim= 16  (trained 10/10 new seeds)  mean improvement=+0.695%  win-rate=50/50 (100.0%)

Stage 1 winner: img_proj_dim = 8


In [10]:
print(f'=== Stage 2: img_dropout sweep (img_proj_dim={BEST_PROJ_DIM}, lr=1e-4, weight_decay=1e-5 fixed) ===')

stage2_evals = {}
for dropout in DROPOUTS:
    seed_results, n_trained = run_or_load_config(BEST_PROJ_DIM, dropout, DEFAULT_LR, DEFAULT_WD, SEEDS)
    ev = evaluate_config(base_by_h, seed_results, HORIZONS, SEEDS)
    stage2_evals[dropout] = ev
    print(f"dropout={dropout:>4}  (trained {n_trained}/{len(SEEDS)} new seeds)  "
          f"mean improvement={ev['mean_improvement_pct']:+.3f}%  "
          f"win-rate={ev['total_wins']}/{ev['total_pairs']} ({ev['win_rate']:.1%})")

BEST_DROPOUT = max(stage2_evals, key=lambda k: stage2_evals[k]['mean_improvement_pct'])
print(f'\nStage 2 winner: img_dropout = {BEST_DROPOUT}')


=== Stage 2: img_dropout sweep (img_proj_dim=8, lr=1e-4, weight_decay=1e-5 fixed) ===
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.1_lr0.0001_wd1e-05 seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.1_lr0.0001_wd1e-05 seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


...

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr0.0001_wd1e-05 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr0.0001_wd1e-05 seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

dropout= 0.5  (trained 10/10 new seeds)  mean improvement=+0.750%  win-rate=50/50 (100.0%)

Stage 2 winner: img_dropout = 0.5


In [11]:
print(f'=== Stage 3: lr sweep (img_proj_dim={BEST_PROJ_DIM}, dropout={BEST_DROPOUT}, weight_decay=1e-5 fixed) ===')

stage3_evals = {}
for lr in LRS:
    seed_results, n_trained = run_or_load_config(BEST_PROJ_DIM, BEST_DROPOUT, lr, DEFAULT_WD, SEEDS)
    ev = evaluate_config(base_by_h, seed_results, HORIZONS, SEEDS)
    stage3_evals[lr] = ev
    print(f"lr={lr:.0e}  (trained {n_trained}/{len(SEEDS)} new seeds)  "
          f"mean improvement={ev['mean_improvement_pct']:+.3f}%  "
          f"win-rate={ev['total_wins']}/{ev['total_pairs']} ({ev['win_rate']:.1%})")

BEST_LR = max(stage3_evals, key=lambda k: stage3_evals[k]['mean_improvement_pct'])
print(f'\nStage 3 winner: lr = {BEST_LR:.0e}')


=== Stage 3: lr sweep (img_proj_dim=8, dropout=0.5, weight_decay=1e-5 fixed) ===
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr5e-05_wd1e-05 seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr5e-05_wd1e-05 seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


...

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr0.0003_wd1e-05 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr0.0003_wd1e-05 seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

lr=3e-04  (trained 10/10 new seeds)  mean improvement=+0.763%  win-rate=50/50 (100.0%)

Stage 3 winner: lr = 3e-04


In [12]:
print(f'=== Stage 4: weight_decay sweep (img_proj_dim={BEST_PROJ_DIM}, dropout={BEST_DROPOUT}, lr={BEST_LR:.0e} fixed) ===')

stage4_evals = {}
for wd in WDS:
    seed_results, n_trained = run_or_load_config(BEST_PROJ_DIM, BEST_DROPOUT, BEST_LR, wd, SEEDS)
    ev = evaluate_config(base_by_h, seed_results, HORIZONS, SEEDS)
    stage4_evals[wd] = ev
    print(f"weight_decay={wd:.0e}  (trained {n_trained}/{len(SEEDS)} new seeds)  "
          f"mean improvement={ev['mean_improvement_pct']:+.3f}%  "
          f"win-rate={ev['total_wins']}/{ev['total_pairs']} ({ev['win_rate']:.1%})")

BEST_WD = max(stage4_evals, key=lambda k: stage4_evals[k]['mean_improvement_pct'])
print(f'\nStage 4 winner: weight_decay = {BEST_WD:.0e}')


=== Stage 4: weight_decay sweep (img_proj_dim=8, dropout=0.5, lr=3e-04 fixed) ===
device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr0.0003_wd0.0 seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr0.0003_wd0.0 seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


...

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr0.0003_wd0.0001 seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

device: cuda (NVIDIA A100-SXM4-40GB)  model: FrozenImageOnlyResidualPredictor  params: 1,032,642


dinov2_proj8_dropout0.5_lr0.0003_wd0.0001 seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

weight_decay=1e-04  (trained 10/10 new seeds)  mean improvement=+0.794%  win-rate=50/50 (100.0%)

Stage 4 winner: weight_decay = 0e+00


In [14]:
print('=== Locked hyperparameters for the final DINOv2 image-only residual model ===')
print(f'  encoder       = {ENCODER_NAME}')
print(f'  img_proj_dim  = {BEST_PROJ_DIM}')
print(f'  img_dropout   = {BEST_DROPOUT}')
print(f'  lr            = {BEST_LR:.0e}')
print(f'  weight_decay  = {BEST_WD:.0e}')

final_ev = stage4_evals[BEST_WD]
print(f"\nAt this configuration: mean improvement over CGM+time = {final_ev['mean_improvement_pct']:+.3f}%, "
      f"win-rate = {final_ev['total_wins']}/{final_ev['total_pairs']} ({final_ev['win_rate']:.1%})")

SUMMARY_PATH = RESULTS_DIR / 'hyperparameter_tuning_summary.json'
with open(SUMMARY_PATH, 'w') as f:
    json.dump({
        'seeds': SEEDS,
        'encoder': ENCODER_NAME,
        'search_stages': {
            'stage1_img_proj_dim': {str(k): v for k, v in stage1_evals.items()},
            'stage2_dropout': {str(k): v for k, v in stage2_evals.items()},
            'stage3_lr': {str(k): v for k, v in stage3_evals.items()},
            'stage4_weight_decay': {str(k): v for k, v in stage4_evals.items()},
        },
        'locked_hyperparameters': {
            'img_proj_dim': BEST_PROJ_DIM, 'img_dropout': BEST_DROPOUT,
            'lr': BEST_LR, 'weight_decay': BEST_WD,
        },
        'final_config_eval': final_ev,
    }, f, indent=2)

print(f'\nSaved hyperparameter tuning summary to: {SUMMARY_PATH}')

=== Locked hyperparameters for the final DINOv2 image-only residual model ===
  encoder       = dinov2
  img_proj_dim  = 8
  img_dropout   = 0.5
  lr            = 3e-04
  weight_decay  = 0e+00

At this configuration: mean improvement over CGM+time = +0.802%, win-rate = 50/50 (100.0%)

Saved hyperparameter tuning summary to: /content/drive/MyDrive/dissertation/experiments/6_best_image_encoder/results/tune_dinov2/hyperparameter_tuning_summary.json
